# Bet Against Beta Portfolio #

In [1]:
# Import Libraries

# Data Management
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Statistics
import statsmodels.api as sm

# Handle Files
import sys
import os

# Import Local Functions
sys.path.append(os.path.abspath("../source"))
from other_data_functions import wexp
from portfolios_helper import calculate_analytics

In [3]:
# Import Data
returns = pd.read_csv(r'../additional_data/stocks_returns.csv')
returns.set_index('Date', inplace=True)
returns.index = pd.to_datetime(returns.index)
returns = returns.dropna(axis=1)

returns

In [4]:
# Get the important data for the Risk-Free Rate
rfr = pd.read_csv(r"../additional_data/risk_free_rate.csv")
rfr.set_index('Date', inplace=True)
rfr.index = pd.to_datetime(rfr.index)

# Get the important data for the S&P500
benchmark = pd.read_csv(r'../additional_data/benchmark_returns.csv')
benchmark.set_index('Date', inplace=True)
benchmark.index = pd.to_datetime(benchmark.index)

### Calculate Betas ###

In [5]:
# Calculate the Market Excess Returns
market_premium = benchmark['benchmark_returns'] - rfr['risk_free_rate']
market_premium.name = 'market_excess_returns'

market_premium

In [6]:
# Calculate Stocks Excess Returns
excess_returns = returns.sub(rfr['risk_free_rate'], axis=0)
excess_returns.dropna(inplace = True)

excess_returns

In [7]:
trimmed_returns = excess_returns.loc['1999':'2014']
trimmed_market = market_premium.loc[trimmed_returns.index]

In [8]:
tickers = returns.columns

In [9]:
# We can use StatsModels efficiently to get the betas for the whole history
betas_list = []

# Loop to Obtain Betas and Alpha + Residuals
for ticker in tickers:
    # Define series
    y_series = trimmed_returns[ticker].dropna()
    
    # Set the Window
    window = len(y_series)
    weights = window * wexp(window, window/2)
    
    # Define weights
    model = sm.WLS(y_series, sm.add_constant(trimmed_market), weights=weights)
    results = model.fit()
    
    beta = results.params.iloc[1]
    
    betas_list.append(beta)

# Create Beta Series
betas_series = pd.Series(betas_list, index=tickers)
betas_series.name = 'history_beta'

betas_series.sort_values(ascending=False)

### First Strategy: Long Only Betas ###

In [10]:
long_beta_portfolio_weights = betas_series / betas_series.sum()
long_beta_portfolio_weights.name = 'weights'

long_beta_portfolio_weights

In [11]:
# Portfolio Returns
long_portfolio_returns = returns.loc['2015':] @ long_beta_portfolio_weights
long_portfolio_returns.name = 'long_portfolio_returns'

long_portfolio_returns

In [12]:
long_portfolio_returns.mean()

In [13]:
# Create Plot
plt.figure(figsize=(10, 6))
plt.plot(long_portfolio_returns.cumsum().mul(100), label='Long-Only Portfolio', color='black', alpha=1)

# Config
plt.title('Returns Time Series')
plt.xlabel('Time')
plt.ylabel('Returns (%)')
plt.legend()
plt.grid()

# Show
plt.show() 

### Second Strategy: Betting Agains Beta ###

In [14]:
# Shrinking Betas
shrunk_betas = 0.6 * betas_series + 0.4

In [15]:
shrunk_betas.sort_values(ascending=False)

In [16]:
# Fist, calculate Ranks
ranks = shrunk_betas.rank()

ranks

In [17]:
# Median
median_rank = ranks.median()

median_rank

In [18]:
# Second: define the groups
low_beta = ranks[ranks < median_rank]
high_beta = ranks[ranks >= median_rank]

In [19]:
# Non-scaled weights
z_bar = ranks.mean()

w_low = (z_bar - low_beta).clip(lower=0)
w_low = w_low / w_low.sum()

w_high = (high_beta - z_bar).clip(lower=0)
w_high = w_high / w_high.sum()

In [20]:
w_low

In [21]:
w_high

In [22]:
# Scale to make beta neutral
beta_low = (w_low * shrunk_betas[w_low.index]).sum()
beta_high = (w_high * shrunk_betas[w_high.index]).sum()

print(beta_low)
print(beta_high)

In [23]:
w_low_scaled = w_low / beta_low
w_high_scaled = w_high / beta_high

In [24]:
w_low_scaled

In [25]:
w_high_scaled

In [26]:
bab_portfolio_weights = pd.concat([w_low_scaled, -w_high_scaled])
bab_portfolio_weights.name = 'weights'

bab_portfolio_weights.sort_values(ascending=True)

In [27]:
bab_portfolio_weights.sum().round(3)

In [28]:
# Portfolio Returns
bab_portfolio_returns = returns.loc['2015':] @ bab_portfolio_weights
bab_portfolio_returns.name = 'bab_portfolio_returns'

bab_portfolio_returns

In [29]:
bab_portfolio_returns.mean()

In [30]:
# Create Plot
plt.figure(figsize=(10, 6))
plt.plot(bab_portfolio_returns.cumsum().mul(100), label='Betting-Against-Beta Portfolio', color='black', alpha=1)

# Config
plt.title('Returns Time Series')
plt.xlabel('Time')
plt.ylabel('Returns (%)')
plt.legend()
plt.grid()

# Show
plt.show() 

### Compare the Strategies ###

In [31]:
# Create DataFrame
strategies_df = pd.DataFrame(index = returns.loc['2015':].index)
strategies_df.index.name = 'date'
strategies_df['long_portfolio'] = long_portfolio_returns
strategies_df['bab_portfolio'] = bab_portfolio_returns

strategies_df

In [32]:
# Create Plot
plt.figure(figsize=(10, 6))
plt.plot(strategies_df.cumsum().mul(100), label=strategies_df.columns, alpha=1)

# Config
plt.title('Returns Time Series')
plt.xlabel('Time')
plt.ylabel('Returns (%)')
plt.legend()
plt.grid()

# Show
plt.show() 

In [33]:
# Analytics
analytics = calculate_analytics(strategies_df)

analytics

### Calculate the Portfolio Beta ###

In [34]:
# Create a DataFrame
regression_df = pd.DataFrame()
regression_df['portfolio_excess'] = bab_portfolio_returns - rfr['risk_free_rate'].loc['2015':]
regression_df['market_excess'] = market_premium.loc['2015':]
regression_df.dropna(inplace = True)

regression_df

In [35]:
# Fit the WLS model
window = len(regression_df)

portfolio_model = model = sm.WLS(
    regression_df['portfolio_excess'], 
    sm.add_constant(regression_df['market_excess']), 
    weights=window*wexp(window, window/2)
)

results = model.fit()

print(results.summary())

### Implementing Rebalancing ###

In [36]:
# Function to calculate the weights (as we saw above)
def betting_against_beta_weights(
    betas: pd.Series
) -> pd.Series:
    
    # Drop NANs
    betas = betas.dropna()

    # Shrinkage
    w_shrink = 0.6      # Adjust
    betas_shrunk = w_shrink * betas + (1 - w_shrink) * 1.0

    # Ranking Betas
    ranks = betas_shrunk.rank()
    z_bar = ranks.mean()
    median_rank = ranks.median()

    # Split using the Median
    low_beta = ranks[ranks < median_rank]
    high_beta = ranks[ranks >= median_rank]

    # Calculate Weights
    w_low = (z_bar - low_beta).clip(lower=0)
    w_high = (high_beta - z_bar).clip(lower=0)

    # Standardize Weights
    w_low /= w_low.sum()
    w_high /= w_high.sum()

    # Scale betas so each side has a beta close to 1
    beta_low = (w_low * betas_shrunk[w_low.index]).sum()
    beta_high = (w_high * betas_shrunk[w_high.index]).sum()

    # Standardize Again
    w_low_scaled = w_low / beta_low
    w_high_scaled = w_high / beta_high

    # Concat
    bab_weights = pd.concat([w_low_scaled, -w_high_scaled])

    return bab_weights

In [38]:
# Fortunately, we calculated the betas (with a 252-d history in the previous notebook)
rolling_betas = pd.read_csv(r'../additional_data/capm_rbetas.csv')
rolling_betas.set_index('Date', inplace=True)
rolling_betas.index = pd.to_datetime(rolling_betas.index)
rolling_betas = rolling_betas[returns.columns]

rolling_betas

In [39]:
# Function for the Rolling Weights
def calculate_bab_rolling_weights(
    beta_df, 
    rebalance_days=21
):
    # Rebalancing Dates
    rebalance_dates = beta_df.index[::rebalance_days]

    # List used for storing
    weights_list = []

    # Loop
    for date in rebalance_dates:
        # Betas for each date
        betas_today = beta_df.loc[date]
        
        # Calculate Weights and store them
        weights = betting_against_beta_weights(betas_today)
        weights.name = date
        weights_list.append(weights)

    # Create a DataFrame
    bab_weights_rebalance = pd.DataFrame(weights_list)

    # Reindexing for daily weights
    bab_weights_daily = bab_weights_rebalance.reindex(beta_df.index)

    # Forward Fill
    bab_weights_daily = bab_weights_daily.ffill().fillna(0)

    # Reindexing Columns to have consistency
    bab_weights_daily = bab_weights_daily.reindex(columns=beta_df.columns).fillna(0)

    return bab_weights_daily


In [40]:
# Get the Weights
bab_daily_weights = calculate_bab_rolling_weights(rolling_betas, rebalance_days=21)

bab_daily_weights

In [41]:
# Calculate the Portfolio using rebalancing
bab_rebalancing_portfolio = (returns.loc[bab_daily_weights.index] * bab_daily_weights).sum(axis = 1)
bab_rebalancing_portfolio.name = 'bab_rebalancing_portfolio'

bab_rebalancing_portfolio

In [42]:
# Create Plot
plt.figure(figsize=(10, 6))
plt.plot(bab_rebalancing_portfolio.cumsum().mul(100), label='BAB Strategy with Rebalacing', alpha=1)

# Config
plt.title('Returns Time Series')
plt.xlabel('Time')
plt.ylabel('Returns (%)')
plt.legend()
plt.grid()

# Show
plt.show() 

In [43]:
# Create Plot
plt.figure(figsize=(10, 6))
plt.plot(strategies_df.cumsum().mul(100), label=strategies_df.columns, alpha=1)
plt.plot(bab_rebalancing_portfolio.loc['2015':].cumsum().mul(100), label='BAB Strategy with Rebalacing', alpha=1)

# Config
plt.title('Returns Time Series')
plt.xlabel('Time')
plt.ylabel('Returns (%)')
plt.legend()
plt.grid()

# Show
plt.show() 

### What about survivorship bias? ###

Previously, we built a portfolio avoiding survivorship bias and using a larger universe of stocks:

In [44]:
bab_realistic_portfolio = pd.read_csv(r'../additional_data/betting_against_beta_portfolio.csv')
bab_realistic_portfolio.set_index('Date', inplace=True)
bab_realistic_portfolio.index = pd.to_datetime(bab_realistic_portfolio.index)

bab_realistic_portfolio

In [45]:
# Create Plot
plt.figure(figsize=(10, 6))
plt.plot(bab_rebalancing_portfolio.cumsum().mul(100), label='BAB Strategy with SB and rebalancing', alpha=1)
plt.plot(bab_realistic_portfolio.cumsum().mul(100), label='BAB Strategy without SB and rebalancing', alpha=1)

# Config
plt.title('Returns Time Series')
plt.xlabel('Time')
plt.ylabel('Returns (%)')
plt.legend()
plt.grid()

# Show
plt.show() 